In [1]:
import kagglehub
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.stattools import durbin_watson
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
)


import matplotlib.pyplot as plt
import seaborn as sns

# Download latest version
path = kagglehub.dataset_download("blastchar/telco-customer-churn")

print("Path to dataset files:", path)

/Users/matthewfischer/miniforge3/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path to dataset files: /Users/matthewfischer/.cache/kagglehub/datasets/blastchar/telco-customer-churn/versions/1


In [2]:
file_path = os.path.join(path, "WA_Fn-UseC_-Telco-Customer-Churn.csv")

telco_customer_churn_original = pd.read_csv(file_path)
telco_customer_churn = telco_customer_churn_original.copy()

In [3]:
print(telco_customer_churn.dtypes)

customerID           object
gender               object
SeniorCitizen         int64
Partner              object
Dependents           object
tenure                int64
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling     object
PaymentMethod        object
MonthlyCharges      float64
TotalCharges         object
Churn                object
dtype: object


In [4]:
telco_customer_churn = telco_customer_churn.drop(columns=["customerID"])

# change total charges from string to numerical
telco_customer_churn["TotalCharges"] = pd.to_numeric(
    telco_customer_churn["TotalCharges"], errors="coerce"
)

# change binary categorical variables to binary numerical
telco_customer_churn["gender"] = (telco_customer_churn["gender"] == "Female").astype(
    int
)

telco_customer_churn["Partner"] = (telco_customer_churn["Partner"] == "Yes").astype(int)

telco_customer_churn["Dependents"] = (
    telco_customer_churn["Dependents"] == "Yes"
).astype(int)

telco_customer_churn["PhoneService"] = (
    telco_customer_churn["PhoneService"] == "Yes"
).astype(int)

telco_customer_churn["MultipleLines"] = (
    telco_customer_churn["MultipleLines"] == "Yes"
).astype(int)

telco_customer_churn["OnlineSecurity"] = (
    telco_customer_churn["OnlineSecurity"] == "Yes"
).astype(int)

telco_customer_churn["OnlineBackup"] = (
    telco_customer_churn["OnlineBackup"] == "Yes"
).astype(int)

telco_customer_churn["DeviceProtection"] = (
    telco_customer_churn["DeviceProtection"] == "Yes"
).astype(int)

telco_customer_churn["TechSupport"] = (
    telco_customer_churn["TechSupport"] == "Yes"
).astype(int)

telco_customer_churn["StreamingTV"] = (
    telco_customer_churn["StreamingTV"] == "Yes"
).astype(int)

telco_customer_churn["StreamingMovies"] = (
    telco_customer_churn["StreamingMovies"] == "Yes"
).astype(int)

telco_customer_churn["PaperlessBilling"] = (
    telco_customer_churn["PaperlessBilling"] == "Yes"
).astype(int)

telco_customer_churn["Churn"] = (telco_customer_churn["Churn"] == "Yes").astype(int)

# use one-hot encoding for non-binary categorical variables
telco_customer_churn = pd.get_dummies(
    telco_customer_churn,
    columns=["InternetService", "Contract", "PaymentMethod"],
    dtype=int,
)

In [5]:
telco_customer_churn

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,OnlineSecurity,OnlineBackup,DeviceProtection,...,InternetService_DSL,InternetService_Fiber optic,InternetService_No,Contract_Month-to-month,Contract_One year,Contract_Two year,PaymentMethod_Bank transfer (automatic),PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,1,0,1,0,1,0,0,0,1,0,...,1,0,0,1,0,0,0,0,1,0
1,0,0,0,0,34,1,0,1,0,1,...,1,0,0,0,1,0,0,0,0,1
2,0,0,0,0,2,1,0,1,1,0,...,1,0,0,1,0,0,0,0,0,1
3,0,0,0,0,45,0,0,1,0,1,...,1,0,0,0,1,0,1,0,0,0
4,1,0,0,0,2,1,0,0,0,0,...,0,1,0,1,0,0,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,0,0,1,1,24,1,1,1,0,1,...,1,0,0,0,1,0,0,0,0,1
7039,1,0,1,1,72,1,1,0,1,1,...,0,1,0,0,1,0,0,1,0,0
7040,1,0,1,1,11,0,0,1,0,0,...,1,0,0,1,0,0,0,0,1,0
7041,0,1,1,0,4,1,1,0,0,0,...,0,1,0,1,0,0,0,0,0,1


In [6]:
telco_customer_churn = telco_customer_churn.dropna()

In [7]:
# binary outcome
print(telco_customer_churn["Churn"].value_counts())

Churn
0    5163
1    1869
Name: count, dtype: int64


In [8]:
# sufficient sample size
for col in telco_customer_churn_original.columns:
    print(f"--- Value Counts for '{col}' ---")
    print(telco_customer_churn_original[col].value_counts())
    print("\n")

--- Value Counts for 'customerID' ---
customerID
7590-VHVEG    1
3791-LGQCY    1
6008-NAIXK    1
5956-YHHRX    1
5365-LLFYV    1
             ..
9796-MVYXX    1
2637-FKFSY    1
1552-AAGRX    1
4304-TSPVK    1
3186-AJIEK    1
Name: count, Length: 7043, dtype: int64


--- Value Counts for 'gender' ---
gender
Male      3555
Female    3488
Name: count, dtype: int64


--- Value Counts for 'SeniorCitizen' ---
SeniorCitizen
0    5901
1    1142
Name: count, dtype: int64


--- Value Counts for 'Partner' ---
Partner
No     3641
Yes    3402
Name: count, dtype: int64


--- Value Counts for 'Dependents' ---
Dependents
No     4933
Yes    2110
Name: count, dtype: int64


--- Value Counts for 'tenure' ---
tenure
1     613
72    362
2     238
3     200
4     176
     ... 
28     57
39     56
44     51
36     50
0      11
Name: count, Length: 73, dtype: int64


--- Value Counts for 'PhoneService' ---
PhoneService
Yes    6361
No      682
Name: count, dtype: int64


--- Value Counts for 'MultipleLines' --

In [9]:
# independence of observations

telco_customer_churn.columns = telco_customer_churn.columns.str.replace(
    " ", "_"
).str.replace(r"[^a-zA-Z0-9_]", "", regex=True)

print(telco_customer_churn.columns)

target_variable = "Churn"
feature_variables = [
    col for col in telco_customer_churn.columns if col != target_variable
]
formula = f"{target_variable} ~ {' + '.join(feature_variables)}"

model = smf.ols(formula=formula, data=telco_customer_churn).fit()

residuals = model.resid
dw_statistic = durbin_watson(residuals)

print(f"Durbin-Watson Statistic: {dw_statistic:.4f}")

Index(['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure',
       'PhoneService', 'MultipleLines', 'OnlineSecurity', 'OnlineBackup',
       'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies',
       'PaperlessBilling', 'MonthlyCharges', 'TotalCharges', 'Churn',
       'InternetService_DSL', 'InternetService_Fiber_optic',
       'InternetService_No', 'Contract_Monthtomonth', 'Contract_One_year',
       'Contract_Two_year', 'PaymentMethod_Bank_transfer_automatic',
       'PaymentMethod_Credit_card_automatic', 'PaymentMethod_Electronic_check',
       'PaymentMethod_Mailed_check'],
      dtype='object')
Durbin-Watson Statistic: 2.0043


In [10]:
# multicollinearity
correlation_matrix = telco_customer_churn.corr(numeric_only=True)
print(correlation_matrix)

                                         gender  SeniorCitizen   Partner  \
gender                                 1.000000       0.001819  0.001379   
SeniorCitizen                          0.001819       1.000000  0.016957   
Partner                                0.001379       0.016957  1.000000   
Dependents                            -0.010349      -0.210550  0.452269   
tenure                                -0.005285       0.015683  0.381912   
PhoneService                           0.007515       0.008392  0.018397   
MultipleLines                          0.008883       0.142996  0.142561   
OnlineSecurity                         0.016328      -0.038576  0.143346   
OnlineBackup                           0.013093       0.066663  0.141849   
DeviceProtection                       0.000807       0.059514  0.153556   
TechSupport                            0.008507      -0.060577  0.120206   
StreamingTV                            0.007124       0.105445  0.124483   
StreamingMov

In [11]:
X = pd.DataFrame(telco_customer_churn, columns=feature_variables)
y = telco_customer_churn["Churn"]

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Scale the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


def evaluate_model(model, X_train, X_test, y_train, y_test, model_name):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    print(f"\n{model_name}:")
    print(f"MSE: {mse:.2f}")
    print(f"R2 Score: {r2:.2f}")

    for feature, coef in zip(X.columns, model.coef_[0]):
        print(f"{feature}: {coef:.4f}")

    return model, y_pred


logistic_model, logistic_pred = evaluate_model(
    LogisticRegression(solver="lbfgs", C=1.0, random_state=42),
    X_train_scaled,
    X_test_scaled,
    y_train,
    y_test,
    "Logistic Regression",
)


Logistic Regression:
MSE: 0.21
R2 Score: -0.09
gender: 0.0158
SeniorCitizen: 0.0950
Partner: 0.0253
Dependents: -0.1002
tenure: -1.4367
PhoneService: -0.0693
MultipleLines: 0.1569
OnlineSecurity: -0.1584
OnlineBackup: -0.0122
DeviceProtection: 0.0370
TechSupport: -0.1457
StreamingTV: 0.2061
StreamingMovies: 0.2155
PaperlessBilling: 0.1347
MonthlyCharges: -0.5842
TotalCharges: 0.6867
InternetService_DSL: -0.0793
InternetService_Fiber_optic: 0.5492
InternetService_No: -0.5681
Contract_Monthtomonth: 0.3125
Contract_One_year: -0.0606
Contract_Two_year: -0.3064
PaymentMethod_Bank_transfer_automatic: -0.0129
PaymentMethod_Credit_card_automatic: -0.0706
PaymentMethod_Electronic_check: 0.1086
PaymentMethod_Mailed_check: -0.0402


/Users/matthewfischer/miniforge3/lib/python3.12/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/matthewfischer/miniforge3/lib/python3.12/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/matthewfischer/miniforge3/lib/python3.12/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/matthewfischer/miniforge3/lib/python3.12/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: divide by zero encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights
/Users/matthewfischer/miniforge3/lib/python3.12/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: overflow encountered in matmul
  grad[:n_features] = X.T @ 

In [12]:
# calculate classification metrics
accuracy = accuracy_score(y_test, logistic_pred)
precision = precision_score(y_test, logistic_pred)
recall = recall_score(y_test, logistic_pred)
f1 = f1_score(y_test, logistic_pred)

# get probability predictions for ROC-AUC
logistic_prob = logistic_model.predict_proba(X_test_scaled)[:, 1]
roc_auc = roc_auc_score(y_test, logistic_prob)

print("\nLogistic Regression Performance:")
print(f"Accuracy:  {accuracy:.3f}")
print(f"Precision: {precision:.3f}")
print(f"Recall:    {recall:.3f}")
print(f"F1 Score:  {f1:.3f}")
print(f"ROC-AUC:   {roc_auc:.3f}")


Logistic Regression Performance:
Accuracy:  0.788
Precision: 0.623
Recall:    0.516
F1 Score:  0.564
ROC-AUC:   0.832


/Users/matthewfischer/miniforge3/lib/python3.12/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/matthewfischer/miniforge3/lib/python3.12/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/matthewfischer/miniforge3/lib/python3.12/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


The odds ratio of a variable is calculated as e to the power of the coefficient from the logistic regression model. A one-standard-deviation increase in a variable is associated with about a (1 - odds ratio) increase (decrease if odds ratio is below 1) in the odds of churn, holding other variables constant.

A limitation of logistic regression for customer churn is that it assumes a relatively simple relationship between the predictors and the log-odds of churn. Logistic regression may fail to capture relationships that more flexible models can identify. Additionally, the model produced numerical overflow warnings, so likely more feature engineering, cleaning, or pruning is needed.